In [4]:
import pandas as pd 
import numpy as np

In [5]:
df=pd.read_csv('D:\Food Delivery Time Prediction\Data\RawData\Food_Delivery_Times.csv')

X=df.drop(columns=['Delivery_Time_min','Order_ID'],axis=1)
y=df['Delivery_Time_min']


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import root_mean_squared_error,mean_absolute_error,r2_score

In [7]:
num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object']).columns

In [8]:
ordinal_cat=['Traffic_Level','Time_of_Day']
onehot_cat=['Weather','Vehicle_Type']
traffic_mapping = ['Low', 'Medium', 'High']
time_mapping = ['Afternoon', 'Evening', 'Night', 'Morning']

In [9]:
numerical_pipeline=Pipeline(
    steps=[
        ('Imputer',SimpleImputer(strategy='mean')),
        ('Scaling',StandardScaler())
    ]
)
cat_oh_pipeline=Pipeline(
    steps=[
        ('Imputer',SimpleImputer(strategy='most_frequent')),
        ('Onehot',OneHotEncoder(drop='first'))
    ]
)
cat_ordinal_pipeline=Pipeline(
    steps=[
        ('Imputer',SimpleImputer(strategy='most_frequent')),
        ('Ordinal',OrdinalEncoder(categories=[traffic_mapping,time_mapping]))
    ]
)

In [10]:
preprocessor=ColumnTransformer(
    transformers=[
        ('num_features',numerical_pipeline,num_features),
        ('onehot',cat_oh_pipeline,onehot_cat),
        ('ordinal',cat_ordinal_pipeline,ordinal_cat)
    ]
)

In [11]:
X=preprocessor.fit_transform(X)

In [12]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.3,random_state=42)

In [13]:
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(),
    'RandomForest': RandomForestRegressor(),
    'AdaBoost': AdaBoostRegressor(),
    'GradientBoosting': GradientBoostingRegressor(),
    'XGBoost': XGBRegressor(),
    'CatBoost': CatBoostRegressor(verbose=0)
}

In [14]:
def evaluate_model(y_train,y_pred):
    return mean_absolute_error(y_train,y_pred),root_mean_squared_error(y_train,y_pred),r2_score(y_train,y_pred)

In [15]:

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

LinearRegression
Model performance for Training set
- Root Mean Squared Error: 11.0081
- Mean Absolute Error: 6.7332
- R2 Score: 0.7473
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 9.1242
- Mean Absolute Error: 5.9649
- R2 Score: 0.8337


DecisionTree
Model performance for Training set
- Root Mean Squared Error: 0.0000
- Mean Absolute Error: 0.0000
- R2 Score: 1.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 15.4840
- Mean Absolute Error: 10.4067
- R2 Score: 0.5212


RandomForest
Model performance for Training set
- Root Mean Squared Error: 4.5489
- Mean Absolute Error: 3.0181
- R2 Score: 0.9568
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 10.0299
- Mean Absolute Error: 6.8130
- R2 Score: 0.7991


AdaBoost
Model performance for Training set
- Root Mean Squared Error: 15.2334
- Mean Absolute Error: 13.2633
- R2 Score: 0.5161
-----------------

In [16]:
param_grids = {
    "LinearRegression": {
        "fit_intercept": [True, False],
        "positive": [True, False]
    },
    
    "DecisionTree": {
        "max_depth": [None, 5, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": [None, "sqrt", "log2"]
    },
    
    "RandomForest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },
    
    "AdaBoost": {
        "n_estimators": [50, 100, 200, 300],
        "learning_rate": [0.001, 0.01, 0.1, 0.5, 1]
    },
    
    "GradientBoosting": {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.001, 0.01, 0.1, 0.2],
        "max_depth": [3, 5, 7],
       
    },
    
    "XGBoost": {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.001, 0.01, 0.1, 0.2],
        "max_depth": [3, 5, 7, 10],

    },
    
    "CatBoost": {
        "depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "iterations": [200, 500, 800]
    }
}


In [17]:
# Running tuning for all models
best_models = {}
for name, model in models.items():
    print(f"Tuning {name}...")
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grids[name],
        n_iter=20,              # number of combinations to try
        cv=5,                   # 5-fold cross-validation
        scoring="r2",
        n_jobs=-1,
        random_state=42,
        verbose=1
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    print(f"Best Params for {name}: {search.best_params_}")
    print(f"Best CV R²: {search.best_score_}")
    print("-"*50)


Tuning LinearRegression...
Fitting 5 folds for each of 4 candidates, totalling 20 fits


d:\Food Delivery Time Prediction\venv\lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 4 is smaller than n_iter=20. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Params for LinearRegression: {'positive': False, 'fit_intercept': True}
Best CV R²: 0.7387746577783807
--------------------------------------------------
Tuning DecisionTree...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Params for DecisionTree: {'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 5}
Best CV R²: 0.5803286791135296
--------------------------------------------------
Tuning RandomForest...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Params for RandomForest: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 30}
Best CV R²: 0.6960399916869934
--------------------------------------------------
Tuning AdaBoost...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Params for AdaBoost: {'n_estimators': 100, 'learning_rate': 0.01}
Best CV R²: 0.6416020253130046
--------------------------------------------------
Tuning GradientBo

## Out of all these models Linear Regression is better but for more accuracy for non linear data i will choose CatBoost to train my model

In [24]:
best_model=CatBoostRegressor(learning_rate = 0.01, iterations = 800, depth =  4)
best_model.fit(X_train,y_train)
import pickle 
import os

with open(os.path.join(os.getcwd(),'model.pkl'),'wb') as file_obj:
    pickle.dump(best_model,file_obj)

0:	learn: 21.7735798	total: 1.58ms	remaining: 1.26s
1:	learn: 21.6566958	total: 3.12ms	remaining: 1.25s
2:	learn: 21.5415534	total: 4.31ms	remaining: 1.15s
3:	learn: 21.4260343	total: 7.79ms	remaining: 1.55s
4:	learn: 21.3102085	total: 11.5ms	remaining: 1.83s
5:	learn: 21.2093185	total: 12.8ms	remaining: 1.69s
6:	learn: 21.0927383	total: 14ms	remaining: 1.59s
7:	learn: 20.9916661	total: 15.1ms	remaining: 1.49s
8:	learn: 20.8920982	total: 17.1ms	remaining: 1.5s
9:	learn: 20.7858058	total: 18.7ms	remaining: 1.48s
10:	learn: 20.6833079	total: 20.4ms	remaining: 1.47s
11:	learn: 20.5926490	total: 21.5ms	remaining: 1.41s
12:	learn: 20.4866498	total: 22.8ms	remaining: 1.38s
13:	learn: 20.3796741	total: 23.8ms	remaining: 1.34s
14:	learn: 20.2700715	total: 25.2ms	remaining: 1.32s
15:	learn: 20.1635931	total: 26.5ms	remaining: 1.3s
16:	learn: 20.0520596	total: 27.7ms	remaining: 1.27s
17:	learn: 19.9563103	total: 29ms	remaining: 1.26s
18:	learn: 19.8630942	total: 30.2ms	remaining: 1.24s
19:	learn